In [1]:
import pandas as pd
import sqlite3
from pathlib import Path
import sys

DB_PATH = Path("../data/finance.db")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")

In [2]:
schema_sql = Path("../sql/schema.sql").read_text(encoding="utf-8")
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

from bankschema import BANK_CONFIGS, normalize_transactions
conn.executescript(schema_sql)
conn.commit()

In [3]:
raw_path = Path("../data/raw/transactions.csv")
sample_path = Path("../data/sample/sample_transactions.csv")
csv_path = raw_path if raw_path.exists() else sample_path
df = pd.read_csv(csv_path)
try:
    df = normalize_transactions(df, BANK_CONFIGS["bank_with_indicator"])
except: 
    df = normalize_transactions(df, BANK_CONFIGS["generic_signed_amount"])
if "Type" in df.columns:
    tx_type = df["Type"].astype(str).str.strip().str.lower()
    df["amount"] = df["amount"].abs()
    df.loc[tx_type.eq("debit"), "amount"] *= -1
    df.loc[tx_type.eq("credit"), "amount"] = df.loc[tx_type.eq("credit"), "amount"].abs()

df.head()

,date,PostedDate,description,Category,Amount,Type,Account,Balance,Currency,Pending,amount
0,2025-11-01,2025-11-01,ACME PAYROLL,Income,2723.98,Credit,Checking,5923.98,USD,False,2723.98
1,2025-11-02,2025-11-02,RENT PAYMENT,Housing,-1454.52,Debit,Checking,4469.46,USD,True,-1454.52
2,2025-11-04,2025-11-05,ACME PAYROLL,Income,2568.20,Credit,Checking,7037.66,USD,False,2568.20
3,2025-11-06,2025-11-06,ACME PAYROLL,Income,2564.63,Credit,Checking,9602.29,USD,False,2564.63
4,2025-11-08,2025-11-10,SHELL,Transportation,-16.35,Debit,Checking,9585.94,USD,False,-16.35


In [4]:
rules = [
    (r"(?:STARBUCKS|DUNKIN)", "Coffee", "Coffee"),
    (r"(?:UBER|LYFT)", "Uber/Lyft", "Transport"),
    (r"(?:SHELL|EXXON|CHEVRON|QT OUTSIDE)", "Gas Station", "Gas"),
    (r"(?:WALMART|TARGET|AMAZON|TRADER JOES|WHOLE FOODS|QT INSIDE)", "Grocery Store", "Groceries"),
    (r"(?:AMAZON|TIKTOK SHOP|TTS|NEWEGG|ETSY|ALIEXPRESS)", "Shopping", "Retail"),
    (r"(?:NETFLIX|SPOTIFY|HULU|PEACOCK)", "Streaming", "Entertainment"),
    (r"(?:PAYROLL|INCOME|DIRECT DEPOSIT|ACH CREDIT|DEPOSIT)", "Income", "Income"),
    (r"(?:RENT|DEDUCTION|BILT)", "Rent", "Housing"),
    (r"(?:VENMO|ZELLE|CASHAPP|TRANSFER)", "Transfers", "Transfers"),
    (r"(?:INTEREST)", "Interest", "Bills"),
    (r"(?:DEBIT)", "Subscriptions", "Subscriptions"),
    (r"(?:TRANSFER TO CREDIT CARD|TRANSFER TO LOAN|USAA|CAPITAL ONE)", "CC/Loan Payment", "Bills"),
    (r"(?:GOOGLE|APPLE)", "Recurring", "Bills"),
    (r"(?:STATE FARM|ALLSTATE|GEICO)", "Insurance", "Bills"),
    (r"(?:CHIPOTLE|MCDONALD'S|SUBWAY|CANES|BURGER KING|PANDA EXPRESS|WINGSTOP)", "Restaurants", "Food"),
    (r"(?:ENTER NEW VALUES HERE)", "New Values", "Category"),
]

df["merchant_name"] = "Unknown"
df["category_name"] = "Uncategorized"

for pattern, merchant, category in rules:
    mask = df["description"].str.contains(pattern, case=False, regex=True, na=False)
    df.loc[mask, "merchant_name"] = merchant
    df.loc[mask, "category_name"] = category

df[["date", "description", "amount", "merchant_name", "category_name"]].head(42)

,date,description,amount,merchant_name,category_name
0,2025-11-01,ACME PAYROLL,2723.98,Income,Income
1,2025-11-02,RENT PAYMENT,-1454.52,Rent,Housing
2,2025-11-04,ACME PAYROLL,2568.20,Income,Income
3,2025-11-06,ACME PAYROLL,2564.63,Income,Income
4,2025-11-08,SHELL,-16.35,Gas Station,Gas
5,2025-11-11,SHELL,-37.77,Gas Station,Gas
6,2025-11-12,AMAZON,-228.42,Shopping,Retail
7,2025-11-13,TRADER JOES,-184.63,Grocery Store,Groceries
8,2025-11-14,AMAZON,-120.13,Shopping,Retail
9,2025-11-15,STARBUCKS,-18.93,Coffee,Coffee


In [5]:
def upsert_dimension(conn, table, col, values):
    conn.executemany(
        f"INSERT OR IGNORE INTO {table} ({col}) VALUES (?)",
        [(v,) for v in sorted(set(values))]
    )
    conn.commit()

upsert_dimension(conn, "categories", "category_name", df["category_name"].tolist())
upsert_dimension(conn, "merchants", "merchant_name", df["merchant_name"].tolist())

cat_map = dict(conn.execute("SELECT category_name, category_id FROM categories").fetchall())
merch_map = dict(conn.execute("SELECT merchant_name, merchant_id FROM merchants").fetchall())

to_insert=[]
for row in df.itertuples(index=False):
    to_insert.append((
        row.date,
        row.description,
        float(row.amount),
        merch_map.get(row.merchant_name),
        cat_map.get(row.category_name)
    ))

conn.executemany(
    """
    INSERT INTO transactions (transaction_date, description, amount, merchant_id, category_id)
    VALUES (?, ?, ?, ?, ?)
    """,
    to_insert
)
conn.commit()
conn.execute("SELECT COUNT(*) FROM transactions").fetchall()

[(42,)]